# cast-qml-02-kernel-heads

**What this does.** Castings version of weld notebook 02, with identical settings. On the same 25 part-grouped splits as cast-qml-01, it compresses each feature set with PCA to *d* = 4, 6, 8, 10, 12 components (= qubits) and trains six heads on identical inputs:

| head | kernel | tuned on validation parts |
|---|---|---|
| `svm_rbf` | classical RBF | γ (10 values) × C (4) |
| `svm_lin` | linear | C (9) |
| `rf` | random forest, 300 trees | fixed (reference) |
| `qsvm_zz` | fidelity kernel of Qiskit's `zz_feature_map` (full entanglement, reps=2, α=2) | bandwidth c (10) × C (4) |
| `qsvm_prod` | same circuit **without** the ZZ entangling terms (`z_feature_map`) | bandwidth c (10) × C (4) |
| `pqk` | projected quantum kernel: Gaussian kernel on the per-qubit Bloch vectors of the ZZ states | bandwidth c (10) × C (4) |

Quantum and RBF heads get the same 40-point tuning budget. Every SVM uses `class_weight="balanced"`. Hyperparameters and the decision threshold are chosen on the validation parts only; the test parts are used once.

**Feature sources:** `hog` (1,764-d), `frozen` (ImageNet ResNet-18, no fine-tuning), `cnnreal` (ResNet-18 fine-tuned on the training patches). PCA and scaling are fit on each split's training patches only. No synthetic data.

**Quantum simulation:** exact statevectors (noiseless). The numpy implementation is checked against Qiskit in the cell below whenever Qiskit is installed.

**Settings:** Accelerator = None (CPU). Internet = ON only if you want the Qiskit cross-check.
**Input:** the output of cast-qml-01 (`cast-qml-01-cnn-embeddings`) only — do not attach the weld notebooks.
**Outputs:** `heads_perfold.csv` (main results), `bw_sweep.csv` (every bandwidth), `cnn_head_perfold.csv` (copied from 01).
**Runtime:** roughly 1.5–3 h on Kaggle CPU. Progress is saved after every split; if the 12-hour limit is reached, save the version, add its output as an extra input and run again to resume.

In [ ]:
import os, glob, time, shutil, numpy as np, pandas as pd
from itertools import combinations
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, recall_score

def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits[0] if hits else None

META = find("cast_meta.csv")
assert META, "Add the output of cast-qml-01-cnn-embeddings as an input"
IN = os.path.dirname(META)
OUT = "/kaggle/working"
print("input folder:", IN)

SEED = 42
SOURCES = ["hog", "frozen", "cnnreal"]
DIMS = [4, 6, 8, 10, 12]
HEAD_TRAIN = ["real"]
BW = np.geomspace(0.005, 2.0, 10)
GAMMA_D = np.geomspace(0.003, 10.0, 10)
CS = [0.1, 1.0, 10.0, 100.0]
LIN_CS = np.geomspace(1e-3, 1e2, 9)
ZZ_REPS, ALPHA = 2, 2.0
GEO_LAMBDA = 0.1
MAX_MINUTES = 660

In [ ]:
def _bits(n):
    return ((np.arange(2 ** n)[:, None] >> np.arange(n)) & 1).astype(np.float64)

def _wht(psi):
    m, N = psi.shape; h = 1
    while h < N:
        v = psi.reshape(m, N // (2 * h), 2, h)
        psi = np.stack([v[:, :, 0] + v[:, :, 1], v[:, :, 0] - v[:, :, 1]], axis=2).reshape(m, N)
        h *= 2
    return psi / np.sqrt(N)

def zz_states(X, reps=ZZ_REPS, alpha=ALPHA):
    X = np.asarray(X, np.float64); m, n = X.shape; N = 2 ** n; Z = _bits(n)
    ph = X @ Z.T
    if n > 1:
        P = list(combinations(range(n), 2))
        A = np.stack([(np.pi - X[:, i]) * (np.pi - X[:, j]) for i, j in P], 1)
        Q = np.stack([np.logical_xor(Z[:, i], Z[:, j]) for i, j in P], 1).astype(np.float64)
        ph = ph + A @ Q.T
    D = np.exp(1j * alpha * ph); psi = D / np.sqrt(N)
    for _ in range(reps - 1): psi = D * _wht(psi)
    return psi

def fidelity(Sa, Sb): return np.abs(Sa @ Sb.conj().T) ** 2

def prod_kernel(A, B, reps=ZZ_REPS, alpha=ALPHA):
    def qs(X):
        e = np.exp(1j * alpha * X); a = np.full(X.shape, 1 / np.sqrt(2), complex); b = e / np.sqrt(2)
        for _ in range(reps - 1): a, b = (a + b) / np.sqrt(2), e * (a - b) / np.sqrt(2)
        return a, b
    a1, b1 = qs(A); a2, b2 = qs(B); K = np.ones((len(A), len(B)))
    for i in range(A.shape[1]):
        K *= np.abs(np.outer(a1[:, i].conj(), a2[:, i]) + np.outer(b1[:, i].conj(), b2[:, i])) ** 2
    return K

def bloch(S):
    m, N = S.shape; n = int(round(np.log2(N))); out = np.empty((m, n, 3))
    for i in range(n):
        v = S.reshape(m, N >> (i + 1), 2, 1 << i); a, b = v[:, :, 0, :], v[:, :, 1, :]
        c = np.einsum("mij,mij->m", a.conj(), b)
        out[:, i, 0], out[:, i, 1] = 2 * c.real, 2 * c.imag
        out[:, i, 2] = np.einsum("mij,mij->m", a.conj(), a).real - np.einsum("mij,mij->m", b.conj(), b).real
    return out.reshape(m, 3 * n)

def sqdist(A, B):
    return np.maximum((A * A).sum(1)[:, None] + (B * B).sum(1)[None, :] - 2 * A @ B.T, 0)

# !pip install -q qiskit    # optional: uncomment (with Internet ON) to cross-check the simulator against Qiskit
try:
    from qiskit.circuit.library import zz_feature_map, z_feature_map
    from qiskit.quantum_info import Statevector
    Xt = np.random.default_rng(0).normal(size=(4, 5))
    ref = np.array([Statevector(zz_feature_map(5, reps=ZZ_REPS).assign_parameters(x)).data for x in Xt])
    refp = np.array([Statevector(z_feature_map(5, reps=ZZ_REPS).assign_parameters(x)).data for x in Xt])
    print("Qiskit cross-check | ZZ kernel max error:", np.abs(fidelity(ref, ref) - fidelity(zz_states(Xt), zz_states(Xt))).max(),
          "| Z kernel max error:", np.abs(fidelity(refp, refp) - prod_kernel(Xt, Xt)).max())
except ImportError:
    print("Qiskit not installed: skipping the cross-check (run `!pip install qiskit` with Internet ON to include it)")

In [ ]:
def best_threshold(yt, p):
    ts = np.unique(np.concatenate([p, [np.median(p)]]))
    ba = [balanced_accuracy_score(yt, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(ba))])

def test_metrics(yva, sva, yte, ste):
    thr = best_threshold(yva, sva); yp = (ste >= thr).astype(int)
    rec = recall_score(yte, yp, average=None, labels=[0, 1], zero_division=0)
    return dict(auc=roc_auc_score(yte, ste), bal_acc=balanced_accuracy_score(yte, yp), rec_def=rec[1], rec_clean=rec[0], thr=thr)

def fit_svm(Ktr, Kva, Kte, ytr, yva, Cs):
    best = None
    for C in Cs:
        m = SVC(kernel="precomputed", C=C, class_weight="balanced").fit(Ktr, ytr)
        sva = m.decision_function(Kva); a = roc_auc_score(yva, sva)
        if best is None or a > best[0] + 1e-12: best = (a, C, m, sva)
    a, C, m, sva = best
    return a, C, sva, m.decision_function(Kte)

def kstats(K, y):
    n = len(K); off = K[~np.eye(n, dtype=bool)]
    Kc = K - K.mean(0) - K.mean(1)[:, None] + K.mean()
    s = np.where(y == 1, 1.0, -1.0); s = s - s.mean(); Y = np.outer(s, s)
    return dict(off_mean=off.mean(), off_std=off.std(), kta=(Kc * Y).sum() / np.sqrt((Kc * Kc).sum() * (Y * Y).sum()))

def geo_diff(Kc, Kq, lam=GEO_LAMBDA):
    wq, Vq = np.linalg.eigh(Kq); sq = (Vq * np.sqrt(np.clip(wq, 0, None))) @ Vq.T
    wc, Vc = np.linalg.eigh(Kc); wc = np.clip(wc, 0, None)
    M = sq @ ((Vc * (wc / (wc + lam) ** 2)) @ Vc.T) @ sq
    return float(np.sqrt(max(np.linalg.eigvalsh(M).max(), 0)))

def prep_features(Xr_raw, Xc_raw, fit, d):
    sc = StandardScaler().fit(Xr_raw[fit])
    if d == 0:
        Zr = sc.transform(Xr_raw)
        return Zr, Zr[:0], 1.0
    p = PCA(d, random_state=SEED).fit(sc.transform(Xr_raw[fit]))
    sc2 = StandardScaler().fit(p.transform(sc.transform(Xr_raw[fit])))
    Zr = sc2.transform(p.transform(sc.transform(Xr_raw)))
    return Zr, Zr[:0], float(p.explained_variance_ratio_.sum())

In [ ]:
meta = pd.read_csv(META); yr = meta.label.values; n_real = len(yr)
H_real = np.load(os.path.join(IN, "hog_real.npy")).astype(np.float64)
F_real = np.load(os.path.join(IN, "frozen_real.npy")).astype(np.float64)
EMB = sorted(glob.glob(os.path.join(IN, "emb", "emb_r*_f*.npz")))
print(f"{n_real} patches ({yr.sum()} defective) | {len(EMB)} split files")
if os.path.exists(os.path.join(IN, "cnn_head_perfold.csv")):
    shutil.copy(os.path.join(IN, "cnn_head_perfold.csv"), OUT)

prev = find("heads_perfold.csv")
rows = pd.read_csv(prev).to_dict("records") if prev and os.path.dirname(prev) != IN else []
sweep = pd.read_csv(find("bw_sweep.csv")).to_dict("records") if rows and find("bw_sweep.csv") else []
done = {(r["repeat"], r["fold"]) for r in rows}
print("splits already done (resumed from a previous version):", len(done))

In [ ]:
def run_split(path):
    z = np.load(path); rep, k = map(int, os.path.basename(path)[5:-4].replace("r", "").split("_f"))
    fit, val, te = z["fit"], z["val"], z["te"]
    n_add = 0
    TR = {"real": fit}
    y_all = yr
    raw = {"hog": (H_real, H_real[:0]), "frozen": (F_real, F_real[:0]), "cnnreal": (z["cnnreal_real"], z["cnnreal_real"][:0])}
    out, sw = [], []
    for src in SOURCES:
        Xr_raw, Xc_raw = raw[src][0].astype(np.float64), raw[src][1].astype(np.float64)
        for d in [0] + DIMS:
            t1 = time.time()
            Zr, Zc, ev = prep_features(Xr_raw, Xc_raw, fit, d)
            S = np.vstack([Zr, Zc]); dim = S.shape[1]
            D2 = sqdist(S, S)
            kern = {"svm_rbf": [(g, np.exp(-(g / dim) * D2)) for g in GAMMA_D]}
            if d > 0:
                kern["qsvm_zz"], kern["qsvm_prod"], kern["pqk"] = [], [], []
                for c in BW:
                    SV = zz_states(c * S); kern["qsvm_zz"].append((c, fidelity(SV, SV)))
                    kern["qsvm_prod"].append((c, prod_kernel(c * S, c * S)))
                    B = bloch(SV); Db = sqdist(B, B); gb = 1.0 / max(np.mean(Db[np.ix_(fit, fit)]), 1e-12)
                    kern["pqk"].append((c, np.exp(-gb * Db)))
            for ht in (HEAD_TRAIN if d > 0 else ["real"]):
                tr = TR[ht]; ytr, yva, yte = y_all[tr], yr[val], yr[te]
                base = dict(repeat=rep, fold=k, source=src, d=d, head_train=ht, n_train=len(tr), pca_ev=ev)
                chosen = {}
                for name, lst in kern.items():
                    best = None
                    for bw, K in lst:
                        a, C, sva, ste = fit_svm(K[np.ix_(tr, tr)], K[np.ix_(val, tr)], K[np.ix_(te, tr)], ytr, yva, CS)
                        if ht == "real":
                            st = kstats(K[np.ix_(fit, fit)], yr[fit])
                            sw.append(dict(repeat=rep, fold=k, source=src, d=d, model=name, bw=bw, C=C, val_auc=a,
                                           auc=roc_auc_score(yte, ste), **st))
                        if best is None or a > best[0] + 1e-12: best = (a, C, bw, sva, ste, K)
                    a, C, bw, sva, ste, K = best; chosen[name] = K[np.ix_(fit, fit)]
                    out.append(dict(base, model=name, bw=bw, C=C, val_auc=a, **test_metrics(yva, sva, yte, ste),
                                    **kstats(K[np.ix_(fit, fit)], yr[fit])))
                if d > 0:
                    for name in ["qsvm_zz", "qsvm_prod", "pqk"]:
                        r = [o for o in out if o["model"] == name and o["source"] == src and o["d"] == d and o["head_train"] == ht][-1]
                        r["geo_vs_rbf"] = geo_diff(chosen["svm_rbf"], chosen[name])
                    Kl = S @ S.T
                    a, C, sva, ste = fit_svm(Kl[np.ix_(tr, tr)], Kl[np.ix_(val, tr)], Kl[np.ix_(te, tr)], ytr, yva, LIN_CS)
                    out.append(dict(base, model="svm_lin", bw=np.nan, C=C, val_auc=a, **test_metrics(yva, sva, yte, ste)))
                    rf = RandomForestClassifier(300, class_weight="balanced", n_jobs=-1, random_state=SEED).fit(S[tr], ytr)
                    sva, ste = rf.predict_proba(S[val])[:, 1], rf.predict_proba(S[te])[:, 1]
                    out.append(dict(base, model="rf", bw=np.nan, C=np.nan, val_auc=roc_auc_score(yva, sva), **test_metrics(yva, sva, yte, ste)))
        best_line = {m: np.mean([o["auc"] for o in out if o["source"] == src and o["model"] == m and o["head_train"] == "real" and o["d"] > 0])
                     for m in ["svm_rbf", "qsvm_zz", "qsvm_prod", "pqk"]}
        print(f"  rep {rep} fold {k} | {src:7s} | mean test AUC over d (real-only heads): " +
              " ".join(f"{m} {v:.3f}" for m, v in best_line.items()), flush=True)
    return rep, k, out, sw

In [ ]:
T0 = time.time()
for path in EMB:
    rep, k = map(int, os.path.basename(path)[5:-4].replace("r", "").split("_f"))
    if (rep, k) in done: continue
    if (time.time() - T0) / 60 > MAX_MINUTES:
        print("Time budget reached: save this version, then rerun with this output added as an input to resume."); break
    t = time.time()
    _, _, out, sw = run_split(path)
    rows += out; sweep += sw
    pd.DataFrame(rows).to_csv(f"{OUT}/heads_perfold.csv", index=False)
    pd.DataFrame(sweep).to_csv(f"{OUT}/bw_sweep.csv", index=False)
    print(f"rep {rep} fold {k} done in {time.time()-t:.0f}s | total {(time.time()-T0)/60:.1f} min", flush=True)

In [ ]:
df = pd.DataFrame(rows)
print("splits complete:", df.groupby(["repeat", "fold"]).ngroups)
q = df[(df.head_train == "real") & (df.d > 0)]
print("\nTest AUC, mean over splits (rows: source, d; columns: head; real-only training):")
print(q.pivot_table(index=["source", "d"], columns="model", values="auc", aggfunc="mean").round(3))